# Fraud Alert (Portfolio Project)
## Hyperparameter Tuning + Model Validation 

### Problem statement
A fintech company wants to **flag suspicious transactions**. Fraud is rare, so we need a model that:
- performs well on unseen data (model validation)
- can be improved by tuning hyperparameters (grid search)

### What you’ll do
1. Load a real-looking dataset from CSV  
2. Clean + preprocess (missing values, scaling, train/test split)  
3. Train a baseline classifier (SVC)  
4. Validate using cross-validation  
5. Tune hyperparameters with GridSearchCV  
6. Compare baseline vs tuned model and write a recommendation


## Learning outcomes
By the end of this notebook, you should be able to:
- Explain **parameters vs hyperparameters**
- Use **train/test split** and **cross-validation** for validation
- Tune an SVC using **GridSearchCV**
- Compare models using simple metrics and make a decision


### 📊 Dataset Description – Fraud Alert 

| **Column Name**         | **Type**            | **Description** |
|--------------------------|---------------------|-----------------|
| **transaction_id**      | Integer             | Unique identifier for each transaction. Used for reference only and not for model training. |
| **amount_usd**          | Numeric             | Transaction amount in US dollars. Larger amounts may increase fraud risk, especially when combined with other suspicious signals. |
| **hour_of_day**         | Integer (0–23)      | Hour when the transaction occurred. Late-night transactions may carry higher fraud risk. |
| **txn_velocity_1h**     | Integer             | Number of transactions made by the account in the last hour. High velocity can indicate suspicious automated activity. |
| **account_age_days**    | Integer             | Number of days since the account was created. Newer accounts are generally higher risk. |
| **failed_logins_7d**    | Integer             | Number of failed login attempts in the past 7 days. Multiple failed attempts may indicate account compromise attempts. |
| **chargebacks_90d**     | Integer             | Number of disputed transactions in the last 90 days. Strong historical indicator of risky behaviour. |
| **country_mismatch**    | Binary (0/1)        | 1 if the transaction country differs from the account’s registered country; otherwise 0. Geographic inconsistencies increase fraud risk. |
| **is_international**    | Binary (0/1)        | 1 if the transaction is international; otherwise 0. International transactions may carry elevated fraud risk. |
| **device_risk_score**   | Numeric (0–100)     | Risk score assigned to the device used for the transaction. Higher scores indicate greater suspicion. |
| **ip_risk_score**       | Numeric (0–100)     | Risk score associated with the IP address used. High-risk IPs increase fraud probability. |
| **merchant_risk_score** | Numeric (0–100)     | Risk score associated with the merchant. Some merchants historically experience higher fraud rates. |
| **fraud**               | Binary (0/1)        | Target variable. 1 = Fraudulent transaction, 0 = Legitimate transaction. |

## 1) Imports

In [28]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from sklearn.metrics import make_scorer


## 2) Load the dataset (CSV)
This makes the project feel like a real ML workflow.


In [29]:
# If you're running locally, put the CSV in the same folder as this notebook.
df = pd.read_csv("fraud_alert_lite_transactions.csv")

df.head()

,transaction_id,amount_usd,hour_of_day,txn_velocity_1h,account_age_days,failed_logins_7d,chargebacks_90d,country_mismatch,is_international,device_risk_score,ip_risk_score,merchant_risk_score,fraud
0,1,15.89,17,1,305,1,0,0,0,0.9,3.3,14.4,0
1,2,36.32,23,2,2228,1,0,0,0,11.8,12.9,46.6,0
2,3,6.57,15,1,3431,0,0,0,1,61.2,21.5,26.3,0
3,4,91.53,19,0,1262,1,0,0,0,18.7,34.5,20.5,0
4,5,49.93,19,2,1268,0,0,0,0,43.7,8.7,69.7,0


### Quick data check

In [30]:
df.shape

(3500, 13)

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   transaction_id       3500 non-null   int64  
 1   amount_usd           3448 non-null   float64
 2   hour_of_day          3500 non-null   int64  
 3   txn_velocity_1h      3500 non-null   int64  
 4   account_age_days     3500 non-null   int64  
 5   failed_logins_7d     3500 non-null   int64  
 6   chargebacks_90d      3500 non-null   int64  
 7   country_mismatch     3500 non-null   int64  
 8   is_international     3500 non-null   int64  
 9   device_risk_score    3448 non-null   float64
 10  ip_risk_score        3448 non-null   float64
 11  merchant_risk_score  3448 non-null   float64
 12  fraud                3500 non-null   int64  
dtypes: float64(4), int64(9)
memory usage: 355.6 KB


In [32]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
transaction_id,3500.0,1750.500000,1010.507298,1.00,875.750,1750.50,2625.250,3500.00
amount_usd,3448.0,40.608425,37.910027,1.49,16.895,29.90,51.340,431.31
hour_of_day,3500.0,11.746000,6.894526,0.00,6.000,12.00,18.000,23.00
txn_velocity_1h,3500.0,1.217429,1.110569,0.00,0.000,1.00,2.000,6.00
account_age_days,3500.0,1824.534571,1055.754845,2.00,915.250,1810.00,2741.250,3647.00
failed_logins_7d,3500.0,0.381714,0.618850,0.00,0.000,0.00,1.000,3.00
chargebacks_90d,3500.0,0.075714,0.272035,0.00,0.000,0.00,0.000,2.00
country_mismatch,3500.0,0.082000,0.274404,0.00,0.000,0.00,0.000,1.00
is_international,3500.0,0.215714,0.411376,0.00,0.000,0.00,0.000,1.00
device_risk_score,3448.0,25.443097,14.698003,0.00,14.700,25.00,35.300,83.80


### Class balance (fraud is rare)

In [33]:
df["fraud"].unique()

array([0, 1])

In [34]:
df["fraud"].value_counts()

fraud
0    3455
1      45
Name: count, dtype: int64

In [35]:
df["fraud"].value_counts(normalize=True) * 100

fraud
0    98.714286
1     1.285714
Name: proportion, dtype: float64

**Why is Accuracy not the best evaluation metric for classification problems?**

In [36]:
df.isnull().sum()

transaction_id          0
amount_usd             52
hour_of_day             0
txn_velocity_1h         0
account_age_days        0
failed_logins_7d        0
chargebacks_90d         0
country_mismatch        0
is_international        0
device_risk_score      52
ip_risk_score          52
merchant_risk_score    52
fraud                   0
dtype: int64

## 3) Preprocessing
We will:
- Fill missing values with 0 
- Drop `transaction_id` (it’s just an identifier)
- Split train/test (75/25)
- Standardise numeric features


In [48]:
df.isnull().sum()

transaction_id          0
amount_usd             52
hour_of_day             0
txn_velocity_1h         0
account_age_days        0
failed_logins_7d        0
chargebacks_90d         0
country_mismatch        0
is_international        0
device_risk_score      52
ip_risk_score          52
merchant_risk_score    52
fraud                   0
dtype: int64

In [59]:
def process_dataset(dataframe):
    """
    This function takes the dataframe and does the following; 
    - fill missing values with 0
    - Drop transaction Id column
    - Splits the data into train and test (75/25)
    - Standardise numerical features

    Parameter:
    input: Dataframe
    Output: (X_train, y_train), (X_test, y_test)
    """
    # Create a copy of the data
    data = dataframe.copy()

    # Fill mission values with 0
    data = data.fillna(0)

    # Drop transaction Id Column
    data = data.drop(columns=["transaction_id"])

    # Separate the data into X and y
    X = data.drop(columns=["fraud"])
    y = data["fraud"]

    # Scale features (Standardise or Normalise)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Split our data into Train and Test
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.25, random_state=42)

    return (X_train, y_train), (X_test, y_test)

In [49]:
(X_train, y_train), (X_test, y_test) = process_dataset(df)

In [51]:
X_train.shape, y_train.shape

((2625, 11), (2625,))

In [52]:
X_test.shape, y_test.shape

((875, 11), (875,))

## 4) Baseline model: SVC
We start with a baseline Support Vector Classifier.

Let's write a function that trains an SVC model


In [55]:
def train_svc_model(X_train, y_train):
    """
    This function takes X_train and y_train data and trains an SVC model.
    """
    model = SVC(random_state=40, gamma='auto')
    model.fit(X_train, y_train)
    return model

    

In [57]:
svc_baseline = train_svc_model(X_train, y_train)

In [58]:
svc_baseline

SVC(gamma='auto', random_state=40)

## 5) Evaluate baseline on the test set

In [60]:
y_pred_base = svc_baseline.predict(X_test)

In [67]:
confusion_matrix(y_test, y_pred_base)

array([[862,   0],
       [ 13,   0]])

In [69]:
print(classification_report(y_test, y_pred_base))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       862
           1       0.00      0.00      0.00        13

    accuracy                           0.99       875
   macro avg       0.49      0.50      0.50       875
weighted avg       0.97      0.99      0.98       875



/opt/anaconda3/envs/alx2025/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/alx2025/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/alx2025/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result)

In [70]:
roc_auc_score(y_test, y_pred_base)

0.5

## 6) Model validation: compare multiple classifiers with cross-validation
We compare:
- Logistic Regression
- Random Forest
- SVC

Metric: ROC-AUC (useful when fraud is rare).


In [73]:
models = {
    "Logistic Regression": LogisticRegression(), 
    "Random Forest": RandomForestClassifier(), 
    "SVC (baseline)": SVC(random_state=40, gamma='auto')
}


In [77]:
models.items()

dict_items([('Logistic Regression', LogisticRegression()), ('Random Forest', RandomForestClassifier()), ('SVC (baseline)', SVC(gamma='auto', random_state=40))])

In [79]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000,), 
    "Random Forest": RandomForestClassifier(), 
    "SVC (baseline)": SVC(random_state=40, gamma='auto')
}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
    print(f"{name} | ROC-AUC CV average: {scores.mean():4f} ")

Logistic Regression | ROC-AUC CV average: 0.733623 
Random Forest | ROC-AUC CV average: 0.631927 
SVC (baseline) | ROC-AUC CV average: 0.642492 


## 7) Parameters vs Hyperparameters (quick reminder)
- **Parameters** are learned from data during training (e.g., weights)
- **Hyperparameters** are chosen by us before training (e.g., C and gamma for SVC)

Let’s inspect SVC hyperparameters.


In [84]:
list(svc_baseline.get_params().keys())

['C',
 'break_ties',
 'cache_size',
 'class_weight',
 'coef0',
 'decision_function_shape',
 'degree',
 'gamma',
 'kernel',
 'max_iter',
 'probability',
 'random_state',
 'shrinking',
 'tol',
 'verbose']

## 8) Custom scoring: Log Loss 
Log loss penalises confident wrong predictions.

We clip predictions to avoid log(0).  
In GridSearchCV, we set `greater_is_better=False` because lower log loss is better.


In [85]:
def custom_log_loss(y_true, y_pred):
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    return float(np.round(loss, 7))

custom_log_loss(np.array(y_test), np.array(y_pred_base))

0.5131475

## 9) Hyperparameter tuning with GridSearchCV (SVC)
We tune:
- C
- gamma


In [86]:
param_grid = {
    "C": [0.1, 1, 10],
    "gamma": [0.01, 0.1, 1]
}


scorer = make_scorer(custom_log_loss, greater_is_better=True)

grid = GridSearchCV(
    estimator=SVC(),
    param_grid=param_grid, 
    scoring=scorer,
    cv=5
)

In [87]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': [0.1, 1, 10], 'gamma': [0.01, 0.1, 1]},
             scoring=make_scorer(custom_log_loss, response_method='predict'))

In [88]:
grid.best_estimator_

SVC(C=10, gamma=0.1)

## 10) Evaluate tuned model

In [89]:
best_svc = grid.best_estimator_
y_pred_tuned = best_svc.predict(X_test)

print("=== BASELINE SVC ===")
print("ROC-AUC:", roc_auc_score(y_test, y_pred_base))
print("Custom Log Loss:", custom_log_loss(np.array(y_test), np.array(y_pred_base)))

print("\n=== TUNED SVC ===")
print("ROC-AUC:", roc_auc_score(y_test, y_pred_tuned))
print("Custom Log Loss:", custom_log_loss(np.array(y_test), np.array(y_pred_tuned)))

print("\nTuned Confusion Matrix:\n", confusion_matrix(y_test, y_pred_tuned))
print("\nTuned Classification Report:\n", classification_report(y_test, y_pred_tuned))

=== BASELINE SVC ===
ROC-AUC: 0.5
Custom Log Loss: 0.5131475

=== TUNED SVC ===
ROC-AUC: 0.4988399071925754
Custom Log Loss: 0.5920951

Tuned Confusion Matrix:
 [[860   2]
 [ 13   0]]

Tuned Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99       862
           1       0.00      0.00      0.00        13

    accuracy                           0.98       875
   macro avg       0.49      0.50      0.50       875
weighted avg       0.97      0.98      0.98       875



## 11) Your portfolio write-up 
Write your conclusion in your own words:
1. Did tuning improve performance? How do you know?
2. What trade-off do you notice (precision vs recall)?
3. If this were a real system, what would you improve next?

**Optional next steps**
- Use probabilities: `SVC(probability=True)` and compute log loss on predicted probabilities
- Try threshold tuning (when do we raise an alert?)
- Add a cost metric (missing fraud is more expensive than a false alert)
